# Step 2b: Population health metrics
Cost (PMPM), hospital use (admits, readmits), high-cost members, chronic conditions.

In [1]:
import duckdb
from pathlib import Path

BASE = Path.home() / "Desktop/projects/claims_analysis"
con = duckdb.connect(str(BASE / "data/claims.duckdb"))

## 1. One cost table for all claims
Stacks the 4 claims files into one long table: member, year, setting, amount paid. Carrier claims have no claim total, so we add up the 13 line payments.

In [2]:
car_paid = " + ".join(f"COALESCE(LINE_NCH_PMT_AMT_{i}, 0)" for i in range(1, 14))

con.execute(f"""
CREATE OR REPLACE TABLE claims_cost AS
SELECT DESYNPUF_ID, year(CLM_FROM_DT) AS year, 'Inpatient' AS setting, CLM_PMT_AMT AS paid FROM inpatient
UNION ALL
SELECT DESYNPUF_ID, year(CLM_FROM_DT), 'Outpatient', CLM_PMT_AMT FROM outpatient
UNION ALL
SELECT DESYNPUF_ID, year(CLM_FROM_DT), 'Professional', {car_paid} FROM carrier
UNION ALL
SELECT DESYNPUF_ID, year(SRVC_DT), 'Pharmacy', TOT_RX_CST_AMT FROM rx
""")

con.sql("""
SELECT setting, COUNT(*) AS claims, ROUND(SUM(paid) / 1e6, 1) AS paid_millions
FROM claims_cost GROUP BY setting ORDER BY paid_millions DESC
""").df()

,setting,claims,paid_millions
0,Inpatient,66494,634.1
1,Professional,4745914,415.1
2,Pharmacy,5561154,341.0
3,Outpatient,792562,225.5


## 2. Member-year table
One row per member per year: months covered, age, sex, state, and cost by setting. This table feeds PMPM, high-cost analysis, the risk model, and Power BI.

In [3]:
con.execute("""
CREATE OR REPLACE TABLE member_year AS
WITH cost AS (
    SELECT DESYNPUF_ID, year,
        SUM(paid) FILTER (WHERE setting = 'Inpatient')    AS ip_paid,
        SUM(paid) FILTER (WHERE setting = 'Outpatient')   AS op_paid,
        SUM(paid) FILTER (WHERE setting = 'Professional') AS prof_paid,
        SUM(paid) FILTER (WHERE setting = 'Pharmacy')     AS rx_paid
    FROM claims_cost GROUP BY DESYNPUF_ID, year
)
SELECT m.DESYNPUF_ID, m.year,
    GREATEST(m.BENE_HI_CVRAGE_TOT_MONS, m.BENE_SMI_CVRAGE_TOT_MONS) AS member_months,
    m.year - year(m.birth_dt) AS age,
    CASE m.BENE_SEX_IDENT_CD WHEN 1 THEN 'Male' ELSE 'Female' END AS sex,
    m.SP_STATE_CODE AS state,
    COALESCE(c.ip_paid, 0)   AS ip_paid,
    COALESCE(c.op_paid, 0)   AS op_paid,
    COALESCE(c.prof_paid, 0) AS prof_paid,
    COALESCE(c.rx_paid, 0)   AS rx_paid,
    COALESCE(c.ip_paid, 0) + COALESCE(c.op_paid, 0)
      + COALESCE(c.prof_paid, 0) + COALESCE(c.rx_paid, 0) AS total_paid
FROM members m
LEFT JOIN cost c USING (DESYNPUF_ID, year)
""")

con.sql("SELECT * FROM member_year LIMIT 5").df()

,DESYNPUF_ID,year,member_months,age,sex,state,ip_paid,op_paid,prof_paid,rx_paid,total_paid
0,17C309D8ED870285,2010,12,84,Female,42,0.0,2760.0,280.0,0.0,3040.0
1,17D4A21C71BFFABF,2009,12,27,Female,52,0.0,20.0,3280.0,450.0,3750.0
2,17DF5512167A95ED,2009,12,81,Female,52,0.0,530.0,1350.0,710.0,2590.0
3,17E09500CD53D265,2008,12,36,Female,10,61000.0,2520.0,6720.0,3900.0,74140.0
4,17E7B78364B91E7A,2008,0,93,Male,39,0.0,760.0,1880.0,4910.0,7550.0


## 3. PMPM (per member per month)
Total paid ÷ total member-months, overall and by setting.

In [4]:
con.sql("""
SELECT year,
    COUNT(*) AS members,
    SUM(member_months) AS member_months,
    ROUND(SUM(total_paid) / SUM(member_months), 2) AS pmpm_total,
    ROUND(SUM(ip_paid)    / SUM(member_months), 2) AS pmpm_inpatient,
    ROUND(SUM(op_paid)    / SUM(member_months), 2) AS pmpm_outpatient,
    ROUND(SUM(prof_paid)  / SUM(member_months), 2) AS pmpm_professional,
    ROUND(SUM(rx_paid)    / SUM(member_months), 2) AS pmpm_pharmacy
FROM member_year
GROUP BY year ORDER BY year
""").df()

,year,members,member_months,pmpm_total,pmpm_inpatient,pmpm_outpatient,pmpm_professional,pmpm_pharmacy
0,2008,116395,1303805.0,458.78,195.61,56.47,112.71,93.98
1,2009,114618,1301064.0,486.76,188.97,67.80,127.07,102.92
2,2010,112845,1281741.0,284.92,101.04,37.71,80.21,65.95


## 4. Hospital stays
A long stay can be billed in more than one piece (`SEGMENT`). This rolls them up to one row per stay.

In [5]:
con.execute("""
CREATE OR REPLACE TABLE stays AS
SELECT DESYNPUF_ID, CLM_ID,
    MIN(COALESCE(CLM_ADMSN_DT, CLM_FROM_DT))       AS admit_dt,
    MAX(COALESCE(NCH_BENE_DSCHRG_DT, CLM_THRU_DT)) AS discharge_dt,
    ANY_VALUE(CLM_DRG_CD)     AS drg,
    ANY_VALUE(ICD9_DGNS_CD_1) AS principal_dx,
    SUM(CLM_PMT_AMT)          AS paid
FROM inpatient
GROUP BY DESYNPUF_ID, CLM_ID
""")

## 5. Admissions per 1,000
Admits ÷ member-years × 1,000. Member-years = member-months ÷ 12.

In [6]:
con.sql("""
WITH adm AS (SELECT year(admit_dt) AS year, COUNT(*) AS admits FROM stays GROUP BY 1),
     mm  AS (SELECT year, SUM(member_months) AS member_months FROM member_year GROUP BY 1)
SELECT year, admits, member_months,
    ROUND(admits * 1000.0 / (member_months / 12.0), 1) AS admits_per_1000
FROM adm JOIN mm USING (year) ORDER BY year
""").df()

,year,admits,member_months,admits_per_1000
0,2008,27449,1303805.0,252.6
1,2009,25264,1301064.0,233.0
2,2010,13514,1281741.0,126.5


## 6. 30-day readmissions
For each stay, find the same member's next admission (`LEAD`). If it starts within 30 days of discharge, the stay counts as readmitted. Stays after Dec 1, 2010 are left out because we can't see 30 days past them.

In [7]:
con.execute("""
CREATE OR REPLACE TABLE readmits AS
SELECT *,
    LEAD(admit_dt) OVER (PARTITION BY DESYNPUF_ID ORDER BY admit_dt) AS next_admit_dt,
    date_diff('day', discharge_dt,
        LEAD(admit_dt) OVER (PARTITION BY DESYNPUF_ID ORDER BY admit_dt)) AS days_to_next
FROM stays
""")

con.sql("""
SELECT year(discharge_dt) AS year,
    COUNT(*) AS index_stays,
    SUM(CASE WHEN days_to_next BETWEEN 0 AND 30 THEN 1 ELSE 0 END) AS readmitted,
    ROUND(100.0 * AVG(CASE WHEN days_to_next BETWEEN 0 AND 30 THEN 1 ELSE 0 END), 1) AS readmit_rate_pct
FROM readmits
WHERE discharge_dt <= DATE '2010-12-01'
GROUP BY 1 ORDER BY 1
""").df()

,year,index_stays,readmitted,readmit_rate_pct
0,2008,27264,3989.0,14.6
1,2009,25282,2021.0,8.0
2,2010,13611,612.0,4.5


## 7. High-cost members
What share of all spending goes to the costliest 5% and 20% of members?

In [8]:
con.sql("""
WITH ranked AS (
    SELECT *, PERCENT_RANK() OVER (PARTITION BY year ORDER BY total_paid DESC) AS pct_rank
    FROM member_year
)
SELECT year,
    ROUND(100.0 * SUM(total_paid) FILTER (WHERE pct_rank < 0.05) / SUM(total_paid), 1) AS top5_share_pct,
    ROUND(100.0 * SUM(total_paid) FILTER (WHERE pct_rank < 0.20) / SUM(total_paid), 1) AS top20_share_pct,
    ROUND(100.0 * AVG((total_paid = 0)::INT), 1) AS pct_zero_cost
FROM ranked GROUP BY year ORDER BY year
""").df()

,year,top5_share_pct,top20_share_pct,pct_zero_cost
0,2008,39.8,72.3,13.7
1,2009,32.5,65.4,11.1
2,2010,34.2,65.3,11.3


## 8. Chronic conditions
Turns the 11 `SP_` flag columns into rows (1 = has it), then shows prevalence and average cost with vs without each condition.

In [10]:
con.execute("""
CREATE OR REPLACE TABLE chronic_long AS
SELECT DESYNPUF_ID, year, condition, flag = 1 AS has_condition
FROM (
    UNPIVOT members
    ON SP_ALZHDMTA, SP_CHF, SP_CHRNKIDN, SP_CNCR, SP_COPD, SP_DEPRESSN,
       SP_DIABETES, SP_ISCHMCHT, SP_OSTEOPRS, SP_RA_OA, SP_STRKETIA
    INTO NAME condition VALUE flag
)
""")

con.sql("""
SELECT c.condition,
    ROUND(100.0 * AVG(c.has_condition::INT), 1) AS prevalence_pct,
    ROUND(AVG(my.total_paid) FILTER (WHERE c.has_condition), 0)     AS avg_cost_with,
    ROUND(AVG(my.total_paid) FILTER (WHERE NOT c.has_condition), 0) AS avg_cost_without
FROM chronic_long c JOIN member_year my USING (DESYNPUF_ID, year)
WHERE c.year = 2010
GROUP BY c.condition ORDER BY prevalence_pct DESC
""").df()

,condition,prevalence_pct,avg_cost_with,avg_cost_without
0,SP_ISCHMCHT,37.6,5426.0,1916.0
1,SP_DIABETES,29.1,6022.0,2091.0
2,SP_CHF,25.8,6009.0,2273.0
3,SP_DEPRESSN,17.9,5735.0,2692.0
4,SP_ALZHDMTA,16.8,5821.0,2714.0
5,SP_CHRNKIDN,13.9,8483.0,2392.0
6,SP_OSTEOPRS,13.1,5210.0,2938.0
7,SP_RA_OA,9.7,5933.0,2946.0
8,SP_COPD,8.9,7918.0,2778.0
9,SP_CNCR,5.2,6570.0,3054.0
